# Time evolution of $\mathrm{Yb}$ spin-orbit coupled system with 2-body loss

- hamiltonian

$$
\begin{align}
    H&=\sum_{\mathbf{k}}[t_{\uparrow\uparrow}c_{\uparrow}^\dagger(\mathbf{k})c_{\uparrow}(\mathbf{k})+t_{\uparrow\downarrow}c_{\uparrow}^\dagger(\mathbf{k})c_{\downarrow}(\mathbf{k})+t_{\downarrow\uparrow}c_{\downarrow}^\dagger(\mathbf{k})c_{\uparrow}(\mathbf{k})+t_{\downarrow\downarrow}c_{\downarrow}^\dagger(\mathbf{k})c_{\downarrow}(\mathbf{k})]-i\frac{\gamma}{2}\sum_{\mathbf{r}}n_\uparrow(\mathbf{r}) n_\downarrow(\mathbf{r}) \\
    &=\underbrace{\sum_{\mathbf{k}}\left[\varepsilon_1(\mathbf{k})c_1^\dagger(\mathbf{k})c_1(\mathbf{k}) + \varepsilon_2(\mathbf{k})c_2^\dagger(\mathbf{k})c_2(\mathbf{k})\right]}_{H_0} + \underbrace{\frac{-1}{N}\frac{i}{2}\gamma\sum_{\mathbf{k}, \mathbf{k}', \mathbf{q}}c_\downarrow^\dagger(\mathbf{k}'+\mathbf{q})c_\uparrow^\dagger(\mathbf{k}-\mathbf{q})c_\uparrow(\mathbf{k})c_\downarrow(\mathbf{k}')}_{V_\text{loss}}
\end{align}

$$

$$t_{\uparrow\uparrow}=\frac{\hbar^2(k-q)^2}{2m_\mathrm{Yb}}+\frac{\delta}{2},\quad t_{\uparrow\downarrow}= t_{\downarrow\uparrow}=\frac{\Omega_R}{2},\quad t_{\downarrow\downarrow}=\frac{\hbar^2(k+q)^2}{2m_\mathrm{Yb}}-\frac{\delta}{2}$$

$$
\begin{pmatrix}
    c_1(\mathbf{k}) \\ c_2(\mathbf{k})
\end{pmatrix}
=
\begin{pmatrix}
    \alpha(\mathbf{k}) & \beta(\mathbf{k}) \\
    -\beta(\mathbf{k}) & \alpha(\mathbf{k})
\end{pmatrix}
\begin{pmatrix}
    c_\uparrow(\mathbf{k}) \\ c_\downarrow(\mathbf{k})
\end{pmatrix},\quad
\begin{pmatrix}
    c_\uparrow(\mathbf{k}) \\ c_\downarrow(\mathbf{k})
\end{pmatrix}
=
\begin{pmatrix}
    \alpha(\mathbf{k}) & -\beta(\mathbf{k}) \\
    \beta(\mathbf{k}) & \alpha(\mathbf{k})
\end{pmatrix}
\begin{pmatrix}
    c_1(\mathbf{k}) \\ c_2(\mathbf{k})
\end{pmatrix}
$$


$$\alpha(\mathbf{k}) = \cos\theta(\mathbf{k}),\quad\beta(\mathbf{k}) = \sin\theta(\mathbf{k}),\quad\theta(\mathbf k)=\frac{1}{2}\arctan\frac{\Omega_R}{\frac{\hbar^2}{2m_\mathrm{Yb}}((\mathbf{k}-\mathbf{q})^2-(\mathbf{k}+\mathbf{q})^2) + \delta}$$


- fourier transforms

$$\begin{align}
	c_\sigma(\mathbf{r})=\frac{1}{\sqrt{N}}\sum_{\mathbf{k}}e^{i\mathbf{k}\cdot{\mathbf{r}}}c_\sigma(\mathbf{k}),&\quad c_\sigma^\dagger(\mathbf{r})=\frac{1}{\sqrt{N}}\sum_{\mathbf{k}}e^{-i\mathbf{k}\cdot{\mathbf{r}}}c_\sigma^\dagger(\mathbf{k})\\
	c_\sigma(\mathbf{k})=\frac{1}{\sqrt{N}}\sum_{\mathbf{r}}e^{-i\mathbf{k}\cdot{\mathbf{r}}}c_\sigma(\mathbf{r}),&\quad c_\sigma^\dagger(\mathbf{k})=\frac{1}{\sqrt{N}}\sum_{\mathbf{r}}e^{i\mathbf{k}\cdot{\mathbf{r}}}c_\sigma^\dagger(\mathbf{r})
\end{align}$$

- basis

$$\mathcal{B}= \{c_{\sigma_1}^\dagger(\mathbf{k}_1)c_{\sigma_2}^\dagger(\mathbf{k}_2)\cdots c_{\sigma_n}^\dagger(\mathbf{k}_n)\ket{0}:\sigma_j=\uparrow,\downarrow, i_{\mathbf{k}_1}<i_{\mathbf{k}_2}<\cdots<i_{\mathbf{k}_n}\}$$
$$[H]_{\mathcal{B}}$$

In [ ]:
import os
import time
import itertools
import functools
import operator
import pickle
from pprint import pprint

import jax
jax.config.update("jax_enable_x64", True)
from jax.extend.backend import get_backend
import jax.numpy as jnp

import numpy as np
import tqdm
import matplotlib.pyplot as plt

from ybsoc import *

In [ ]:
system = YbSOC2bodyLoss(
    d=1,
    lengths=[8],
    n_particle=6,
    hbar=1,
    q=0,# 0.03,
    m_Yb = 1,
    delta = 0,# 0.08,
    omega_R = 0.1,
    gamma = 0.3,
    array_type='numpy'
)
hamiltonian = system.dense_hamiltonian(
    display_progress=True
)

NameError: name 'gamma' is not defined

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(hamiltonian)

In [ ]:
initial_state = system.get_momentum_eigenstate(
    (
        ((0,), 1),
        ((1,), -1),
    )
)


psi_tilde = np.linalg.solve(eigenvectors, initial_state)

t0 = 0
t1 = 100
step_size = 1 / 6
save_at = np.arange(t0, t1, step_size)

evolution_operator_diag = np.exp(-1j * eigenvalues * step_size / system.hbar)
psis = np.zeros((len(save_at), len(psi_tilde)), dtype=np.complex128)

psis[0, :] = np.einsum('ij,j->i', eigenvectors, psi_tilde)
for i in range(len(save_at)):
    psi_tilde = evolution_operator_diag * psi_tilde
    psis[i, :] = np.einsum('ij,j->i', eigenvectors, psi_tilde)
    
    
spin_up_numbers, spin_down_numbers = system.momentum_sp_expected_numbers_vectorized(psis)

In [ ]:
plt.figure(figsize=(6, 5))
t = save_at

# for i in range(8):
#     plt.plot(t, spin_up_numbers[:, i], label=f"$\\uparrow$, {i}")
#     plt.plot(t, spin_down_numbers[:, i], label=f"$\\downarrow$, {i}")

total_up = np.sum(spin_up_numbers, axis=1)
total_down = np.sum(spin_down_numbers, axis=1)

plt.plot(t, total_up, label="$\\uparrow$")
plt.plot(t, total_down, label="$\\downarrow$")
plt.plot(t, total_up + total_down, label="$\\uparrow + \\downarrow$")

plt.ylabel("$\\langle n \\rangle$")
plt.xlabel("$t$")

plt.legend()
plt.show()

In [ ]:
from matplotlib.animation import FFMpegWriter

def make_video(t, up_nums, down_nums, output_path, fps=60):
    """
    a_data, b_data: shape = (num_frames, N)
                    시간축으로 변화하는 a, b 데이터
    output_path: 출력할 mp4 경로
    fps: 초당 프레임 수
    """
    num_frames = len(t)
    N = up_nums.shape[1]
    
    # x축 index
    index = np.arange(N)
    # 막대 너비
    width = ((6 - 1) / N) * 0.75 / 2

    # ---------- (1) 초기 그래프 설정 ----------
    # 첫 프레임(0) 기준으로 막대그래프를 그려두고, 참조를 저장해 놓습니다.
    up_num = up_nums[0]
    down_num = down_nums[0]
    total_num = up_num + down_num

    fig, ax = plt.subplots(figsize=(6, 4))

    # 배경(a+b) 막대: alpha=0.2, color='green', 너비=2*width
    bar_total = ax.bar(index, total_num, width=2*width, alpha=0.2, color='green')

    # a, b 막대: grouped bar
    bar_up = ax.bar(index - width/2, up_num, width, label='up')
    bar_down = ax.bar(index + width/2, down_num, width, label='down')

    ax.set_xticks(index)
    ax.set_xlabel('Index')
    ax.set_ylabel('$\\langle n\\rangle$')
    ax.set_title('t=0.00')
    ax.legend()

    # ---------- (2) 프레임 업데이트 함수 정의 ----------
    def update(frame):
        # 해당 frame에서의 a, b
        up_num = up_nums[frame]
        down_num = down_nums[frame]
        total_num = up_num + down_num
        
        # 배경(a+b) 막대 높이 갱신
        for rect, h in zip(bar_total, total_num):
            rect.set_height(h)
        
        # a 막대 높이 갱신
        for rect, h in zip(bar_up, up_num):
            rect.set_height(h)
        
        # b 막대 높이 갱신
        for rect, h in zip(bar_down, down_num):
            rect.set_height(h)
        
        # 예: 프레임 번호 혹은 시간에 따라 타이틀 갱신
        ax.set_title(f't={t[frame]:.2f}')

    # ---------- (3) mp4로 저장 ----------
    writer = FFMpegWriter(fps=fps, metadata=dict(artist='Me'), bitrate=1800)
    with writer.saving(fig, output_path, dpi=100):
        # 전체 프레임 순회하며 그린 뒤 저장
        for frame in tqdm.trange(num_frames, desc="making video"):
            update(frame)
            writer.grab_frame()
    
    plt.close(fig)

In [ ]:
from IPython.display import Video

video_path = "/home/pco0511/yb-soc-two-body-loss/visualizations/output.mp4"

make_video(t, spin_up_numbers, spin_down_numbers, video_path)
Video(video_path, embed=True)